# Notebook 1: Data Cleaning: BudgetWise Finance Dataset
**File:** `budgetwise_finance_dataset.csv`  
**Goal:** Clean the raw dataset and export a clean CSV ready for SQL analysis  
**Tool:** Python (Pandas)

---
### Data Quality Issues Found
- Mixed date formats across rows
- Category names with typos, mixed casing and misspellings
- Payment mode inconsistencies and abbreviations
- Location names with abbreviations and mixed casing
- Amount column stored as string with currency symbols and impossible values
- Missing values in date, category, amount, payment_mode and location
- Duplicate rows and duplicate transaction IDs

## Step 1: Import Libraries

In [66]:
import pandas as pd
import numpy as np
import re

## Step 2: Load the Raw Dataset

In [67]:
df = pd.read_csv('C:\\Users\\Chibueze\\Valentine-Python-Project\\PythonProject\\Practical\\BudgetWise Personal Finance Dataset\\data\\raw\\budgetwise_finance_dataset.csv')

df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN
...,...,...,...,...,...,...,...,...,...
15895,T7467,U033,21-07-24,Expense,Rnt,7335,Upi,hyderabad,Coffee
15896,T7486,U142,27-02-24,Expense,Educaton,4353,card,BANGALORE,NaN
15897,T13247,U001,03/04/2024,Expense,food,1048,CRD,BANGALORE,Train ticket
15898,T13301,U147,2022-07-09,Income,Others,621176,CARD,CHENNAI,Restaurant dinner


In [68]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15900 entries, 0 to 15899
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    15900 non-null  object
 1   user_id           15900 non-null  object
 2   date              15414 non-null  object
 3   transaction_type  15900 non-null  object
 4   category          15615 non-null  object
 5   amount            15609 non-null  object
 6   payment_mode      15092 non-null  object
 7   location          14638 non-null  object
 8   notes             13079 non-null  object
dtypes: object(9)
memory usage: 1.1+ MB


(15900, 9)

## Step 3: Initial Data Overview

In [69]:
# Check data types

print(' Data Types ')
print(df.dtypes)

print('\n Missing Values ')
print(df.isnull().sum())

print('\n Missing Values % ')
print((df.isnull().sum() / len(df) * 100).round(2))

 Data Types 
transaction_id      object
user_id             object
date                object
transaction_type    object
category            object
amount              object
payment_mode        object
location            object
notes               object
dtype: object

 Missing Values 
transaction_id         0
user_id                0
date                 486
transaction_type       0
category             285
amount               291
payment_mode         808
location            1262
notes               2821
dtype: int64

 Missing Values % 
transaction_id       0.00
user_id              0.00
date                 3.06
transaction_type     0.00
category             1.79
amount               1.83
payment_mode         5.08
location             7.94
notes               17.74
dtype: float64


## Step 4: Remove Duplicate Rows

In [70]:
# Remove fully duplicate rows

df = df.drop_duplicates().copy()

In [71]:
# Remove duplicate transaction_ids and keep first occurrence

df = df.drop_duplicates(subset='transaction_id', keep='first')

In [72]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
Index: 14000 entries, 0 to 15899
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    14000 non-null  object
 1   user_id           14000 non-null  object
 2   date              13569 non-null  object
 3   transaction_type  14000 non-null  object
 4   category          13748 non-null  object
 5   amount            13746 non-null  object
 6   payment_mode      13306 non-null  object
 7   location          12894 non-null  object
 8   notes             11528 non-null  object
dtypes: object(9)
memory usage: 1.1+ MB


(14000, 9)

In [73]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN
...,...,...,...,...,...,...,...,...,...
15893,T12021,U030,05/31/2021,Expense,Traval,3074,CRD,PUNE,NaN
15894,T13908,U113,21-02-2023,Expense,Fod,$9952,NaN,LUC,NaN
15895,T7467,U033,21-07-24,Expense,Rnt,7335,Upi,hyderabad,Coffee
15897,T13247,U001,03/04/2024,Expense,food,1048,CRD,BANGALORE,Train ticket


## Step 5: Clean the Date Column
**Issues:** Mixed formats like `2023-04-25`, `08/05/2022`, `31-12-23` all in the same column

In [74]:
print('Sample date values before cleaning:')
print(df['date'].dropna().unique()[:15])

Sample date values before cleaning:
['2023-04-25' '08/05/2022' '31-12-23' '10/28/2022' '04/11/2024'
 '08/16/2022' '2024-12-12' '13-03-24' '11/10/2023' '2022-04-15'
 '2021-04-18' '06/29/2022' '02/24/2022' '08/28/2023' '02-06-23']


In [75]:
# Parse all mixed date formats into one standard format

def parse_mixed_dates(date_str):
    if pd.isnull(date_str):
        return pd.NaT
    
    formats = [
        '%Y-%m-%d',    # 2023-04-25
        '%m/%d/%Y',    # 10/28/2022
        '%d/%m/%Y',    # 08/05/2022
        '%d-%m-%y',    # 31-12-23
        '%d-%m-%Y',    # 31-12-2023
        '%m-%d-%Y',    # 08-05-2022
        '%Y/%m/%d',    # 2023/04/25
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    return pd.NaT

df['date'] = df['date'].apply(parse_mixed_dates)

print('Rows with unparseable dates after fix:', df['date'].isna().sum())
print('Rows remaining:', len(df))

Rows with unparseable dates after fix: 431
Rows remaining: 14000


In [76]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,2022-08-05,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,2023-12-31,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaT,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,2022-10-28,Expense,entertainment,2287,CARD,Hyderabad,NaN
...,...,...,...,...,...,...,...,...,...
15893,T12021,U030,2021-05-31,Expense,Traval,3074,CRD,PUNE,NaN
15894,T13908,U113,2023-02-21,Expense,Fod,$9952,NaN,LUC,NaN
15895,T7467,U033,2024-07-21,Expense,Rnt,7335,Upi,hyderabad,Coffee
15897,T13247,U001,2024-03-04,Expense,food,1048,CRD,BANGALORE,Train ticket


In [77]:
# Parse all mixed date formats into one standard format with coercion for errors

df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [78]:
# Rows where date could not be parsed become NaT flag them

print('\nRows with unparseable dates (NaT):', df['date'].isna().sum())


Rows with unparseable dates (NaT): 431


In [79]:
# Drop rows where date is still missing cause date is critical for trend analysis

df = df.dropna(subset=['date']).copy()
print('Rows after dropping missing dates:', len(df))

Rows after dropping missing dates: 13569


In [80]:
# Standardise to YYYY-MM-DD

df['date'] = df['date'].dt.strftime('%Y-%m-%d')

In [81]:
# Add helper columns for year, month, and month name for easier analysis later

df['year']  = pd.to_datetime(df['date']).dt.year

df['month'] = pd.to_datetime(df['date']).dt.month

df['month_name'] = pd.to_datetime(df['date']).dt.strftime('%B')

In [82]:
print('\nSample dates after cleaning:')
print(df['date'].unique()[:10])


Sample dates after cleaning:
['2023-04-25' '2022-08-05' '2023-12-31' '2022-10-28' '2024-04-11'
 '2022-08-16' '2024-12-12' '2024-03-13' '2023-11-10' '2022-04-15']


In [83]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,rent,649,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,entertainment,2287,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Foods,4168,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15893,T12021,U030,2021-05-31,Expense,Traval,3074,CRD,PUNE,NaN,2021,5,May
15894,T13908,U113,2023-02-21,Expense,Fod,$9952,NaN,LUC,NaN,2023,2,February
15895,T7467,U033,2024-07-21,Expense,Rnt,7335,Upi,hyderabad,Coffee,2024,7,July
15897,T13247,U001,2024-03-04,Expense,food,1048,CRD,BANGALORE,Train ticket,2024,3,March


## Step 6: Clean the Amount Column
**Issues:** Stored as string, currency symbols (₹), commas, negative values, impossible values (999999, 999999999, 0)

In [84]:
print('Sample amount values before cleaning:')
print(df['amount'].value_counts().head(15))

Sample amount values before cleaning:
amount
999999        71
-500          60
-1000         55
999999999     52
0             50
₹999999999    13
₹999999        9
2407           6
1969           6
5749           6
1759           6
8065           6
179            5
4398           5
4280           5
Name: count, dtype: int64


In [85]:
# Remove currency symbols and commas from amount column

df['amount'] = df['amount'].astype(str).str.replace('₹', '', regex=False)

df['amount'] = df['amount'].str.replace(',', '', regex=False)

df['amount'] = df['amount'].str.strip()

df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,rent,649,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,entertainment,2287,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Foods,4168,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15893,T12021,U030,2021-05-31,Expense,Traval,3074,CRD,PUNE,NaN,2021,5,May
15894,T13908,U113,2023-02-21,Expense,Fod,$9952,NaN,LUC,NaN,2023,2,February
15895,T7467,U033,2024-07-21,Expense,Rnt,7335,Upi,hyderabad,Coffee,2024,7,July
15897,T13247,U001,2024-03-04,Expense,food,1048,CRD,BANGALORE,Train ticket,2024,3,March


In [86]:
# Convert to numeric then non-numeric values become NaN

df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

In [87]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Educaton,3888.0,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,rent,649.0,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239.0,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,entertainment,2287.0,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Foods,4168.0,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15893,T12021,U030,2021-05-31,Expense,Traval,3074.0,CRD,PUNE,NaN,2021,5,May
15894,T13908,U113,2023-02-21,Expense,Fod,NaN,NaN,LUC,NaN,2023,2,February
15895,T7467,U033,2024-07-21,Expense,Rnt,7335.0,Upi,hyderabad,Coffee,2024,7,July
15897,T13247,U001,2024-03-04,Expense,food,1048.0,CRD,BANGALORE,Train ticket,2024,3,March


In [88]:
# Remove impossible values with negative amounts, zero and outlier sentinel values

print('\nRows with negative amounts:', (df['amount'] < 0).sum())
print('Rows with zero amount:', (df['amount'] == 0).sum())
print('Rows with sentinel value 999999:', (df['amount'] == 999999).sum())
print('Rows with sentinel value 999999999:', (df['amount'] == 999999999).sum())


Rows with negative amounts: 125
Rows with zero amount: 55
Rows with sentinel value 999999: 80
Rows with sentinel value 999999999: 65


In [89]:
df = df[df['amount'] > 0]

df = df[df['amount'] < 999999]


In [90]:
# Drop rows where amount is still null

df = df.dropna(subset=['amount']).copy()

df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Educaton,3888.0,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,rent,649.0,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239.0,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,entertainment,2287.0,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Foods,4168.0,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619.0,Cash,delhi,NaN,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travl,3041.0,Bank Transfr,PUN,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Traval,3074.0,CRD,PUNE,NaN,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rnt,7335.0,Upi,hyderabad,Coffee,2024,7,July


In [91]:
# Cast to integer

df['amount'] = df['amount'].astype(int)

In [92]:
print('\nAmount after cleaning:')
print(df['amount'].describe())


Amount after cleaning:
count     11614.000000
mean      15831.753315
std       50381.699497
min          54.000000
25%        2932.000000
50%        5940.000000
75%        9476.000000
max      990020.000000
Name: amount, dtype: float64


In [93]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,rent,649,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,entertainment,2287,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Foods,4168,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619,Cash,delhi,NaN,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travl,3041,Bank Transfr,PUN,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Traval,3074,CRD,PUNE,NaN,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rnt,7335,Upi,hyderabad,Coffee,2024,7,July


## Step 7: Clean the Category Column
**Issues:** Typos, mixed casing, misspellings like `Food`, `FOOD`, `food`, `Fod`, `Fodd`, `Foods` for the same category

In [94]:
df['category'].unique()

array(['Educaton', 'rent', 'Freelance', 'entertainment', 'Foods',
       'Salary', 'Utilties', 'Others', 'Utlities', 'Rentt', 'Food',
       'FOOD', 'Travel', 'food', 'Entertainment', 'Travl', 'education',
       'Investment', 'Foodd', 'HEALTH', 'Helth', 'Education', 'utilities',
       'Rent', 'savings', 'EDU', 'Traval', nan, 'health', 'Bonus',
       'Utility', 'travel', 'Utilities', 'Rnt', 'Entertain', 'Saving',
       'Entrtnmnt', 'TRAVEL', 'RENT', 'SAVINGS', 'Misc', 'others', 'Fod',
       'Health', 'Other', 'Savings', 'OTHERS'], dtype=object)

In [95]:
print('Unique category count before cleaning:', df['category'].nunique())
print('Category values before cleaning:')
print(df['category'].value_counts().to_string())

Unique category count before cleaning: 46
Category values before cleaning:
category
Food             430
FOOD             427
Foods            418
Others           416
RENT             413
food             409
Foodd            405
Rent             384
rent             382
Rentt            381
Fod              381
Rnt              377
Investment       356
Freelance        355
Salary           342
Bonus            331
Travel           302
TRAVEL           296
travel           290
Traval           288
Travl            279
Entertain        257
Utility          254
entertainment    254
Utilties         241
Entrtnmnt        238
Entertainment    226
Utlities         221
Utilities        221
utilities        209
education        208
Education        193
EDU              176
Educaton         175
Health           117
health           115
Helth            110
HEALTH           108
Savings           80
SAVINGS           74
savings           71
Saving            59
Other             39
Misc         

In [96]:
# Standardise to title case and strip whitespace first

df['category'] = df['category'].astype(str).str.strip().str.title()

In [97]:
print('Unique category count before cleaning:', df['category'].nunique())
df['category'].unique()

Unique category count before cleaning: 32


array(['Educaton', 'Rent', 'Freelance', 'Entertainment', 'Foods',
       'Salary', 'Utilties', 'Others', 'Utlities', 'Rentt', 'Food',
       'Travel', 'Travl', 'Education', 'Investment', 'Foodd', 'Health',
       'Helth', 'Utilities', 'Savings', 'Edu', 'Traval', 'Nan', 'Bonus',
       'Utility', 'Rnt', 'Entertain', 'Saving', 'Entrtnmnt', 'Misc',
       'Fod', 'Other'], dtype=object)

In [98]:
# Map all variations to standard names

category_mapping = {
    # Food variations
    'Foods'        : 'Food',
    'Fod'          : 'Food',
    'Foodd'        : 'Food',

    # Rent variations
    'Rentt'        : 'Rent',
    'Rnt'          : 'Rent',

    # Travel variations
    'Traval'       : 'Travel',
    'Travl'        : 'Travel',

    # Entertainment variations
    'Entertain'    : 'Entertainment',
    'Entrtnmnt'    : 'Entertainment',

    # Utilities variations
    'Utility'      : 'Utilities',
    'Utilties'     : 'Utilities',
    'Utlities'     : 'Utilities',

    # Education variations
    'Educaton'     : 'Education',
    'Edu'          : 'Education',

    # Health variations
    'Helth'        : 'Health',

    # Savings variations
    'Saving'       : 'Savings',

    # Others variations
    'Other'        : 'Others',
    'Misc'         : 'Others',

    # Income categories
    'Freelance'    : 'Freelance',
    'Bonus'        : 'Bonus',
    'Investment'   : 'Investment',
}

df['category'] = df['category'].replace(category_mapping)

In [99]:
# Replace the string 'Nan' with real NaN

df['category'] = df['category'].replace('Nan', np.nan)

In [100]:
# Now replace real NaN with 'Unknown'

df['category'] = df['category'].fillna('Unknown')

In [101]:
print('Unique category count After cleaning:', df['category'].nunique())
print('Category values before cleaning:')
print(df['category'].value_counts().to_string())

Unique category count After cleaning: 14
Category values before cleaning:
category
Food             2470
Rent             1937
Travel           1455
Utilities        1146
Entertainment     975
Education         752
Others            552
Health            450
Investment        356
Freelance         355
Salary            342
Bonus             331
Savings           284
Unknown           209


In [102]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Education,3888,card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,Rent,649,NaN,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Csh,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,Entertainment,2287,CARD,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Food,4168,NaN,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619,Cash,delhi,NaN,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travel,3041,Bank Transfr,PUN,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Travel,3074,CRD,PUNE,NaN,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rent,7335,Upi,hyderabad,Coffee,2024,7,July


## Step 8: Clean the Payment Mode Column
**Issues:** Same value written many ways like `UPI`, `Upi`, `upi`, `UPi`, `Card`, `CARD`, `Csh`, `Bank Transfr`, `Bank_Transfer`

In [103]:
df['payment_mode'].unique()

array(['card', nan, 'Csh', 'CARD', 'CRD', 'upi', 'UPI', 'Bank Transfer',
       'bank transfer', 'UPi', 'BankTransfer', 'Bank_Transfer', 'Card',
       'Bank Transfr', 'Cash', 'Crd', 'Upi', 'csh', 'cash', 'CASH'],
      dtype=object)

In [104]:
print('Payment mode before cleaning:')
print(df['payment_mode'].value_counts().to_string())

Payment mode before cleaning:
payment_mode
UPI              700
Upi              689
upi              675
UPi              634
Bank Transfr     590
CARD             581
card             570
csh              567
cash             563
Card             560
Bank Transfer    558
CRD              555
bank transfer    553
Bank_Transfer    553
Csh              545
Cash             544
Crd              536
BankTransfer     535
CASH             531


In [105]:
# Standardise to title case and strip whitespace

df['payment_mode'] = df['payment_mode'].str.strip().str.title()

In [106]:
print('Unique payment mode count before cleaning:', df['payment_mode'].nunique())
df['payment_mode'].unique()

Unique payment mode count before cleaning: 9


array(['Card', nan, 'Csh', 'Crd', 'Upi', 'Bank Transfer', 'Banktransfer',
       'Bank_Transfer', 'Bank Transfr', 'Cash'], dtype=object)

In [107]:
# Map all variations to 4 standard values

payment_mapping = {
    # UPI variations
    'Upi'           : 'UPI',

    # Card variations
    'Crd'           : 'Card',

    # Cash variations
    'Csh'           : 'Cash',

    # Bank Transfer variations
    'Bank Transfr'  : 'Bank Transfer',
    'Banktransfer'  : 'Bank Transfer',
    'Bank_Transfer' : 'Bank Transfer',
}

df['payment_mode'] = df['payment_mode'].replace(payment_mapping)

In [108]:
# Replace the string 'Nan' with real NaN

df['payment_mode'] = df['payment_mode'].replace('Nan', np.nan)

In [109]:
# Now replace real NaN with 'Unknown'

df['payment_mode'] = df['payment_mode'].fillna('Unknown')

In [110]:
print('Payment mode after cleaning:')
print(df['payment_mode'].value_counts().to_string())

Payment mode after cleaning:
payment_mode
Card             2802
Bank Transfer    2789
Cash             2750
UPI              2698
Unknown           575


In [111]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Education,3888,Card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,Rent,649,Unknown,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Cash,BAN,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,Entertainment,2287,Card,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Food,4168,Unknown,NaN,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619,Cash,delhi,NaN,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travel,3041,Bank Transfer,PUN,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Travel,3074,Card,PUNE,NaN,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rent,7335,UPI,hyderabad,Coffee,2024,7,July


## Step 9: Clean the Location Column
**Issues:** Abbreviations (`BAN`, `HYD`, `PUN`), mixed casing, inconsistent naming

In [112]:
df['location'].unique()

array(['Ahmedabad', 'Hyderabad', 'BAN', nan, 'AHM', 'KOL', 'LUC',
       'BANGALORE', 'CHENNAI', 'PUNE', 'lucknow', 'Bangalore', 'Jaipur',
       'JAI', 'hyderabad', 'LUCKNOW', 'CHE', 'JAIPUR', 'Delhi', 'KOLKATA',
       'AHMEDABAD', 'DELHI', 'MUM', 'Pune', 'DEL', 'mumbai', 'chennai',
       'Mumbai', 'kolkata', 'Kolkata', 'bangalore', 'ahmedabad', 'MUMBAI',
       'jaipur', 'HYDERABAD', 'Lucknow', 'Chennai', 'delhi', 'HYD', 'PUN',
       'pune'], dtype=object)

In [113]:
print('Location before cleaning:')
print(df['location'].value_counts().to_string())

Location before cleaning:
location
PUN          295
BANGALORE    293
PUNE         290
chennai      287
HYD          282
KOL          282
kolkata      282
Chennai      281
LUCKNOW      280
lucknow      277
DEL          277
Jaipur       276
mumbai       274
AHM          272
LUC          271
Kolkata      271
Pune         271
ahmedabad    269
delhi        269
BAN          269
Ahmedabad    268
hyderabad    268
AHMEDABAD    265
Mumbai       265
MUMBAI       265
KOLKATA      265
Delhi        259
Bangalore    259
DELHI        259
pune         259
JAIPUR       258
JAI          256
MUM          255
bangalore    251
HYDERABAD    251
jaipur       248
CHENNAI      245
Hyderabad    241
CHE          240
Lucknow      237


In [114]:
# Standardise to title case and strip whitespace

df['location'] = df['location'].str.strip().str.title()

In [115]:
df['location'].unique()

array(['Ahmedabad', 'Hyderabad', 'Ban', nan, 'Ahm', 'Kol', 'Luc',
       'Bangalore', 'Chennai', 'Pune', 'Lucknow', 'Jaipur', 'Jai', 'Che',
       'Delhi', 'Kolkata', 'Mum', 'Del', 'Mumbai', 'Hyd', 'Pun'],
      dtype=object)

In [116]:
# Map abbreviations and variations to full standard city names

location_mapping = {
    # Mumbai
    'Mum'       : 'Mumbai',

    # Delhi
    'Del'       : 'Delhi',

    # Bangalore
    'Ban'       : 'Bangalore',
    'Bengaluru' : 'Bangalore',

    # Hyderabad
    'Hyd'       : 'Hyderabad',

    # Chennai
    'Che'       : 'Chennai',

    # Kolkata
    'Kol'       : 'Kolkata',

    # Pune
    'Pun'       : 'Pune',

    # Ahmedabad
    'Ahm'       : 'Ahmedabad',

    # Jaipur
    'Jai'       : 'Jaipur',

    # Lucknow
    'Luc'       : 'Lucknow',
}

df['location'] = df['location'].replace(location_mapping)

In [117]:
# Replace the string 'Nan' with real NaN

df['location'] = df['location'].replace('Nan', np.nan)

In [118]:
# Now replace real NaN with 'Unknown'

df['location'] = df['location'].fillna('Unknown')

In [119]:
print('Location after cleaning:')
print(df['location'].value_counts().to_string())

Location after cleaning:
location
Pune         1115
Kolkata      1100
Ahmedabad    1074
Bangalore    1072
Lucknow      1065
Delhi        1064
Mumbai       1059
Chennai      1053
Hyderabad    1042
Jaipur       1038
Unknown       932


In [120]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Education,3888,Card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,Rent,649,Unknown,Hyderabad,asdfgh,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Cash,Bangalore,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,Entertainment,2287,Card,Hyderabad,NaN,2022,10,October
5,T12465,U042,2024-04-11,Expense,Food,4168,Unknown,Unknown,test,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619,Cash,Delhi,NaN,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travel,3041,Bank Transfer,Pune,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Travel,3074,Card,Pune,NaN,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rent,7335,UPI,Hyderabad,Coffee,2024,7,July


## Step 10: Clean the Notes Column
**Issues:** Junk entries like `asdfgh`, leading/trailing whitespace, missing values

In [121]:
print('Unique category count before cleaning:', df['notes'].nunique())
print('Notes before cleaning:')
print(df['notes'].value_counts().to_string())

Unique category count before cleaning: 26
Notes before cleaning:
notes
Electricity bill        425
Monthly rent payment    404
Gym membership          399
ATM withdrawal          398
Movie tickets           395
Grocery shopping        389
Restaurant dinner       383
...                     379
Coffee                  379
!!!                     375
Internet bill           373
Doctor consultation     373
test                    365
xyz123                  364
Uber ride               359
Online course           357
Lunch                   357
Petrol                  354
Medicine                351
asdfgh                  351
EMI payment             348
misc                    348
Books                   346
Train ticket            343
Fixed deposit           339
Shopping                333


In [122]:
# Strip whitespace

df['notes'] = df['notes'].astype(str).str.strip()

In [123]:
# Replace junk entries — single characters or random keyboard strings

junk_pattern = r'^[a-zA-Z]{1,3}$|^[a-z]+[0-9]+$|^asdf.*$|^qwer.*$|^zxcv.*$'

df['notes'] = df['notes'].replace(junk_pattern, 'N/A', regex=True)

In [124]:
# Replace specific junk values with N/A
junk_values = ['test', '...', '!!!', 'misc', 'Misc', 'MISC']
df['notes'] = df['notes'].replace(junk_values, 'N/A')

In [125]:
print('Notes After cleaning:')
print(df['notes'].value_counts().to_string())

Notes After cleaning:
notes
N/A                     4209
Electricity bill         425
Monthly rent payment     404
Gym membership           399
ATM withdrawal           398
Movie tickets            395
Grocery shopping         389
Restaurant dinner        383
Coffee                   379
Internet bill            373
Doctor consultation      373
Uber ride                359
Online course            357
Lunch                    357
Petrol                   354
Medicine                 351
EMI payment              348
Books                    346
Train ticket             343
Fixed deposit            339
Shopping                 333


In [126]:
df

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Education,3888,Card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,Rent,649,Unknown,Hyderabad,N/A,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Cash,Bangalore,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,Entertainment,2287,Card,Hyderabad,N/A,2022,10,October
5,T12465,U042,2024-04-11,Expense,Food,4168,Unknown,Unknown,N/A,2024,4,April
...,...,...,...,...,...,...,...,...,...,...,...,...
15891,T0122,U104,2022-05-14,Expense,Health,619,Cash,Delhi,N/A,2022,5,May
15892,T10396,U014,2023-07-22,Expense,Travel,3041,Bank Transfer,Pune,ATM withdrawal,2023,7,July
15893,T12021,U030,2021-05-31,Expense,Travel,3074,Card,Pune,N/A,2021,5,May
15895,T7467,U033,2024-07-21,Expense,Rent,7335,UPI,Hyderabad,Coffee,2024,7,July


## Step 11: Final Data Type Check and Validation

In [127]:
# Convert date to datetime

df['date'] = pd.to_datetime(df['date'])

In [128]:
# Convert low cardinality text columns to category dtype

df['transaction_type'] = df['transaction_type'].astype('category')

df['category']         = df['category'].astype('category')

df['payment_mode']     = df['payment_mode'].astype('category')

df['location']         = df['location'].astype('category')

In [129]:
# Verify

print(' Final Data Types ')
print(df.dtypes)

print('\n Final Missing Values ')
print(df.isnull().sum())

print('\n Final Shape ')
print(df.shape)

print('\n Final Sample ')
df.head(10)

 Final Data Types 
transaction_id              object
user_id                     object
date                datetime64[ns]
transaction_type          category
category                  category
amount                       int64
payment_mode              category
location                  category
notes                       object
year                         int32
month                        int32
month_name                  object
dtype: object

 Final Missing Values 
transaction_id      0
user_id             0
date                0
transaction_type    0
category            0
amount              0
payment_mode        0
location            0
notes               0
year                0
month               0
month_name          0
dtype: int64

 Final Shape 
(11614, 12)

 Final Sample 


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes,year,month,month_name
0,T4999,U018,2023-04-25,Expense,Education,3888,Card,Ahmedabad,Movie tickets,2023,4,April
1,T12828,U133,2022-08-05,Expense,Rent,649,Unknown,Hyderabad,N/A,2022,8,August
2,T7403,U091,2023-12-31,Income,Freelance,13239,Cash,Bangalore,Books,2023,12,December
4,T7495,U088,2022-10-28,Expense,Entertainment,2287,Card,Hyderabad,N/A,2022,10,October
5,T12465,U042,2024-04-11,Expense,Food,4168,Unknown,Unknown,N/A,2024,4,April
7,T9824,U061,2024-12-12,Income,Salary,62061,Card,Ahmedabad,N/A,2024,12,December
8,T0741,U053,2024-03-13,Expense,Utilities,5070,UPI,Kolkata,Grocery shopping,2024,3,March
9,T12403,U004,2023-11-10,Income,Others,59543,UPI,Lucknow,Gym membership,2023,11,November
12,T9446,U006,2022-06-29,Expense,Utilities,5767,Bank Transfer,Bangalore,N/A,2022,6,June
13,T3016,U099,2022-02-24,Expense,Rent,23760,Card,Bangalore,Electricity bill,2022,2,February


## Step 12: Export Clean Dataset

In [ ]:
df.to_csv('cleaned_finance.csv', index=False)

## Step 13: Loading Cleaned Dataset into Postgresql

In [130]:
from sqlalchemy import create_engine
from sqlalchemy import text

DB_USER     = 'postgres'
DB_PASSWORD = 'Chibueze04'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'budgetwise_db'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Drop the view first so the table can be replaced
with engine.connect() as conn:
    conn.execute(text('DROP VIEW IF EXISTS master_transactions;'))
    conn.commit()
    print('View dropped successfully')

# Now load clean dataframe into PostgreSQL
df.to_sql(
    name      = 'budgetwise_finance',
    con       = engine,
    if_exists = 'replace',
    index     = False
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text('SELECT COUNT(*) FROM budgetwise_finance'))
    print('Rows in budgetwise_finance:', result.scalar())

print('budgetwise_finance loaded into PostgreSQL successfully')

View dropped successfully
Rows in budgetwise_finance: 11614
budgetwise_finance loaded into PostgreSQL successfully
